# Fraud Detection: Preprocessing

## Goal
Prepare the data for modeling by solving three problems 
discovered in EDA:

1. **Scale problem** : Amount and Time have much larger values 
   than V1–V28, which would unfairly dominate the model
2. **Split problem** : We need to ensure fraud is proportionally 
   represented in both train and test sets
3. **Imbalance problem** : Only 492 fraud cases out of 284,807 
   transactions — the model needs to see more fraud examples to 
   learn from

## Key Concept
We fix what the model **learns from** (SMOTE) separately from 
how we **measure** it (Precision/Recall). Both matter.

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE

In [2]:
df = pd.read_csv('/Users/mac/creditcard.csv')
print(df.shape)

(284807, 31)


## Step 1: Scale Amount and Time
V1–V28 are already scaled between -3 and 3 because they 
went through PCA transformation by the bank.

Amount and Time however are raw values:
- Amount ranges from $0 to $25,000
- Time goes up to 172,000 seconds

If we leave them as is, the model will think they are more 
important just because of their size — even if V1–V28 are 
actually more useful for detecting fraud.

StandardScaler transforms each column so the mean becomes 
0 and values spread between roughly -3 and 3 — same range 
as V1–V28. Now every feature gets an equal chance.

In [3]:
scaler = StandardScaler()

df['Amount_Scaled'] = scaler.fit_transform(df[['Amount']])
df['Time_Scaled'] = scaler.fit_transform(df[['Time']])

In [4]:
df.head

<bound method NDFrame.head of             Time         V1         V2        V3        V4        V5  \
0            0.0  -1.359807  -0.072781  2.536347  1.378155 -0.338321   
1            0.0   1.191857   0.266151  0.166480  0.448154  0.060018   
2            1.0  -1.358354  -1.340163  1.773209  0.379780 -0.503198   
3            1.0  -0.966272  -0.185226  1.792993 -0.863291 -0.010309   
4            2.0  -1.158233   0.877737  1.548718  0.403034 -0.407193   
...          ...        ...        ...       ...       ...       ...   
284802  172786.0 -11.881118  10.071785 -9.834783 -2.066656 -5.364473   
284803  172787.0  -0.732789  -0.055080  2.035030 -0.738589  0.868229   
284804  172788.0   1.919565  -0.301254 -3.249640 -0.557828  2.630515   
284805  172788.0  -0.240440   0.530483  0.702510  0.689799 -0.377961   
284806  172792.0  -0.533413  -0.189733  0.703337 -0.506271 -0.012546   

              V6        V7        V8        V9  ...       V23       V24  \
0       0.462388  0.239599  0.

In [5]:
df = df.drop(['Amount', 'Time'], axis=1)
print(df.shape)

(284807, 31)


## Step 2: Separate Features and Target
We split the data into:
- **X** — the inputs (everything except Class). This is the 
  evidence the model uses to make a prediction.
- **y** — the target (Class column only). This is what the 
  model is trying to predict: fraud or not fraud.

Think of X as the evidence and y as the verdict.

In [6]:
X = df.drop('Class', axis=1)  # everything except Class
y = df['Class']                # only Class

print(X.shape)
print(y.shape)

(284807, 30)
(284807,)


## Step 3: Train/Test Split (Stratified)
We split 80% of the data for training and 20% for testing.

We use stratified splitting to ensure the 0.17% fraud ratio 
is preserved in both splits. Without this, we might get very 
few fraud cases in the test set by bad luck, making our 
evaluation unreliable.

Important: We never touch the test set until the final evaluation. 
It represents the real world.

In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Training size: {X_train.shape[0]:,}')
print(f'Test size: {X_test.shape[0]:,}')
print(f'Fraud in training: {y_train.sum()}')
print(f'Fraud in test: {y_test.sum()}')

Training size: 227,845
Test size: 56,962
Fraud in training: 394
Fraud in test: 98


## Step 4: Handle Class Imbalance with SMOTE
Even with the right metrics, the model still needs to learn 
from balanced data. If it only sees 492 fraud examples out 
of 227,845 training rows it will barely learn fraud patterns.

SMOTE (Synthetic Minority Oversampling Technique) generates 
synthetic fraud transactions based on patterns in the existing 
492 real fraud cases until fraud matches the legitimate count.

Critical rule: SMOTE is applied to training data ONLY.
The test set must stay untouched to reflect real world 
conditions where fraud is genuinely 0.17% rare.

In [8]:
smote = SMOTE(random_state=42)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

print(f'Before SMOTE - Fraud: {y_train.sum()} | Legit: {(y_train==0).sum()}')
print(f'After SMOTE  - Fraud: {y_train_sm.sum()} | Legit: {(y_train_sm==0).sum()}')

Before SMOTE - Fraud: 394 | Legit: 227451
After SMOTE  - Fraud: 227451 | Legit: 227451


## Key Takeaways

1. **Scaling**: Amount and Time are now on the same scale 
   as V1–V28. Every feature gets a fair chance.

2. **Stratified split**: Fraud ratio is preserved in both 
   train (0.173%) and test (0.172%) sets.

3. **SMOTE**: Training data is now balanced with 227,451 
   fraud and 227,451 legitimate transactions.

## Next Step
we train three models and compare 
them to find the best fraud detector.